<a href="https://colab.research.google.com/github/csugasi/DATASCI266-Banking77-Encoder-vs-Decoder/blob/shuchi%2Fdev/qwen_decoder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import userdata
import os

token = userdata.get('GH_TOKEN')
repo  = "csugasi/DATASCI266-Banking77-Encoder-vs-Decoder"
path  = "/content/repo"

if not os.path.exists(path):
    !git clone -q -b shuchi/dev https://{token}@github.com/{repo}.git {path}

%cd {path}
!git log --oneline -3
!ls data/splits

/content/repo
4530782 (HEAD -> shuchi/dev, origin/shuchi/dev, origin/main, origin/HEAD) Merge pull request #2 from csugasi/chandana-baseline
9475ccc (origin/chandana-baseline) Update README file
53850a0 Add results notebook
full_train.parquet	 train_k20_seed0.parquet  train_k3_seed2.parquet
manifest.json		 train_k20_seed1.parquet  train_k5_seed0.parquet
train_k10_seed0.parquet  train_k20_seed2.parquet  train_k5_seed1.parquet
train_k10_seed1.parquet  train_k30_seed0.parquet  train_k5_seed2.parquet
train_k10_seed2.parquet  train_k30_seed1.parquet  train_p50_seed0.parquet
train_k1_seed0.parquet	 train_k30_seed2.parquet  train_p50_seed1.parquet
train_k1_seed1.parquet	 train_k3_seed0.parquet   train_p50_seed2.parquet
train_k1_seed2.parquet	 train_k3_seed1.parquet   val.parquet


In [ ]:
import sys, json, torch
from transformers import AutoTokenizer, AutoModelForCausalLM

sys.path.insert(0, "scripts")
from predictions import match_intent, write_predictions

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
tokenizer.padding_side = "left"
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float16, device_map="cuda")
model.eval()

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1536,), eps=1e-06)
    (rotar

In [ ]:
import pandas as pd, json

val   = pd.read_parquet("data/splits/val.parquet")
test  = pd.read_parquet("data/raw/test.parquet")
k5    = pd.read_parquet("data/splits/train_k5_seed0.parquet")
labels = json.load(open("data/raw/label_names.json"))

print("val:", val.shape, "| test:", test.shape, "| k5 seed0:", k5.shape, "| labels:", len(labels))
print(k5.head(3))

val: (385, 3) | test: (3080, 2) | k5 seed0: (385, 3) | labels: 77
   id                                               text  label
0  41     Where is my credit card that was to be mailed?     11
1  59   I want to find out what happened to my new card?     11
2  72  How can I see the progress of delivery of the ...     11


In [ ]:
!nvidia-smi

Sun Nov 15 01:30:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P0             25W /   70W |    3107MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install -q peft bitsandbytes accelerate

import torch, transformers, peft, bitsandbytes, accelerate
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0))
print("transformers", transformers.__version__, "| peft", peft.__version__,
      "| bitsandbytes", bitsandbytes.__version__, "| accelerate", accelerate.__version__)

torch 2.14.1+cu130 | GPU: Tesla T4
transformers 5.22.1 | peft 0.21.2 | bitsandbytes 0.50.2 | accelerate 1.15.0


In [ ]:
labels = json.load(open("data/raw/label_names.json"))

SYSTEM = ("You classify banking customer messages. Reply with exactly one intent "
          "name from the list, copied exactly, and nothing else.")
INTENT_LIST = "\n".join(labels)

def build_prompt(message):
    chat = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": f"Intents:\n{INTENT_LIST}\n\nMessage: {message}\n\nIntent:"},
    ]
    return tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)

def classify(messages, batch_size=16):
    outputs = []
    for i in range(0, len(messages), batch_size):
        prompts = [build_prompt(m) for m in messages[i:i + batch_size]]
        inputs = tokenizer(prompts, return_tensors="pt", padding=True).to("cuda")
        with torch.no_grad():
            generated = model.generate(**inputs, max_new_tokens=20, do_sample=False)
        new_tokens = generated[:, inputs["input_ids"].shape[1]:]
        outputs += tokenizer.batch_decode(new_tokens, skip_special_tokens=True)
    return outputs

example = val["text"].iloc[0]
answer = classify([example])[0]
print("Message:    ", example)
print("Qwen says:  ", repr(answer))
print("True intent:", labels[val["label"].iloc[0]])
print("Matched id: ", match_intent(answer))

Message:     Do you have info about the card on delivery?
Qwen says:   'card_arrival'
True intent: card_arrival
Matched id:  11


In [ ]:
import numpy as np

raw = classify(val["text"].tolist())
pred = np.array([match_intent(r) for r in raw])

print("Validation accuracy:", round((pred == val["label"].to_numpy()).mean(), 3))
print("Unmatched (-1):", (pred == -1).sum(), "of", len(pred))

print("\nExamples of unmatched answers:")
for r in sorted(set(r for r, p in zip(raw, pred) if p == -1))[:15]:
    print("  ", repr(r))

path = write_predictions("qwen_zeroshot", "zeroshot", 0, "val", pred, raw_outputs=raw)
print("\nSaved:", path)

Validation accuracy: 0.387
Unmatched (-1): 25 of 385

Examples of unmatched answers:
   'access_account'
   'add_another_card_to_account'
   'bank_transfer'
   'cheque_or_cash_deposit'
   'create_account_for_child'
   'duplicate_transaction_report'
   'exchange_currency_to_europe'
   'exchange_fiat_currency'
   'fee_charged'
   'get_spare_card'
   'pending_transaction'
   'receive_money'
   'refund_not_showed_up'
   'refund_not_shown_up'
   'revert_card_payment?'

Saved: /content/repo/results/qwen_zeroshot_zeroshot_seed0_val.csv


In [ ]:
import evaluation as ev

df = ev.load_predictions(path)
miss = df[df["predicted_label"] == -1].copy()
miss["true_intent"] = miss["true_label"].map(dict(enumerate(labels)))
miss[["raw_output", "true_intent"]]

,raw_output,true_intent
11,verify_exchange_rate_policy,exchange_rate
23,refund_not_showed_up,extra_charge_on_statement
32,exchange_currency_to_europe,fiat_currency_support
50,exchange_fiat_currency,exchange_via_app
60,unknown,age_limit
61,create_account_for_child,age_limit
128,revert_card_payment?,top_up_reverted
130,cheque_or_cash_deposit,balance_not_updated_after_cheque_or_cash_deposit
191,access_account,passcode_forgotten
202,pending_transaction,pending_card_payment


In [ ]:
# Run zero shot evaluation on test set now:
raw_test = classify(test["text"].tolist())
pred_test = np.array([match_intent(r) for r in raw_test])
print("Test accuracy:", round((pred_test == test["label"].to_numpy()).mean(), 3))

Test accuracy: 0.421


In [ ]:
path_test = write_predictions("qwen_zeroshot", "zeroshot", 0, "test", pred_test, raw_outputs=raw_test)
print(path_test)

/content/repo/results/qwen_zeroshot_zeroshot_seed0_test.csv
